# 26a — Encoder-Decoder Architectures

Corresponds to: Note 26 — Encoder-Decoder Architectures

**Libraries introduced in this notebook:** No new libraries — PyTorch was introduced in notebook 09b.

**What this notebook demonstrates:**
1. Manual encoder-decoder forward pass — reproducing the worked example from Note 26
2. Encoder-decoder in PyTorch — building the two-RNN architecture and verifying shapes
3. Sequence-to-sequence training — teaching the model to reverse a short integer sequence

In [1]:
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
print(f'PyTorch: {torch.__version__}')

PyTorch: 2.14.1+cpu


## Part 1: Manual Encoder-Decoder Forward Pass

Reproducing the Note 26 worked example.
Encoder: same weights as Note 24, input $(1.0, 0.5, -1.0)$, produces $c = h_3^{(e)}$.
Decoder: initialised with $c$, generates two output steps.

In [2]:
# ── Encoder (Note 24 weights) ─────────────────────────────────────────────────
Wx_e = np.array([[0.5], [-0.3]])
Wh_e = np.array([[0.4, 0.1], [-0.2, 0.6]])
bh_e = np.zeros((2, 1))

enc_input = [1.0, 0.5, -1.0]
h = np.zeros((2, 1))

print('Encoder forward pass:')
for t, xt in enumerate(enc_input, 1):
    x = np.array([[xt]])
    z = Wh_e @ h + Wx_e @ x + bh_e
    h = np.tanh(z)
    print(f'  t={t}: h = {h.ravel().round(4)}')

c = h.copy()
print(f'\nContext vector c = h3_enc = {c.ravel().round(4)}')
print('Matches Note 26: c = [-0.3685, -0.0137]')

Encoder forward pass:
  t=1: h = [ 0.4621 -0.2913]
  t=2: h = [ 0.3848 -0.3946]
  t=3: h = [-0.3675 -0.0137]

Context vector c = h3_enc = [-0.3675 -0.0137]
Matches Note 26: c = [-0.3685, -0.0137]


In [3]:
# ── Decoder (Note 26 weights) ─────────────────────────────────────────────────
Wh_d = np.array([[0.3, 0.2], [0.1, 0.4]])
Wy_d = np.array([[0.5], [-0.4]])
bh_d = np.zeros((2, 1))
Wo   = np.array([[0.6, 0.8]])

h_d = c.copy()   # initialise decoder with context vector
y_prev = np.array([[0.0]])   # start token

print('Decoder forward pass (initialised with c):')
for t in range(1, 3):
    z_d = Wh_d @ h_d + Wy_d @ y_prev + bh_d
    h_d = np.tanh(z_d)
    y_hat = Wo @ h_d
    print(f'  t={t}: z={z_d.ravel().round(4)}  h={h_d.ravel().round(4)}  y_hat={y_hat.ravel().round(4)}')
    y_prev = y_hat   # autoregressive: use own prediction as next input

print()
print('Matches Note 26: y_hat1 = -0.1016, y_hat2 = -0.0459')

Decoder forward pass (initialised with c):
  t=1: z=[-0.113  -0.0422]  h=[-0.1125 -0.0422]  y_hat=[-0.1013]
  t=2: z=[-0.0928  0.0124]  h=[-0.0926  0.0124]  y_hat=[-0.0456]

Matches Note 26: y_hat1 = -0.1016, y_hat2 = -0.0459


## Part 2: Encoder-Decoder in PyTorch

Building the two-RNN architecture. The encoder and decoder are separate `nn.RNN` modules with separate weights. The context vector is passed as the decoder's initial hidden state.

In [4]:
class EncoderDecoder(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.encoder = nn.RNN(input_size=input_size,  hidden_size=hidden_size, batch_first=True)
        self.decoder = nn.RNN(input_size=output_size, hidden_size=hidden_size, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def encode(self, src):
        # src: (batch, src_len, input_size)
        _, h_n = self.encoder(src)   # h_n: (1, batch, hidden)
        return h_n                   # context vector

    def decode(self, tgt, context):
        # tgt: (batch, tgt_len, output_size)  context: (1, batch, hidden)
        out, _ = self.decoder(tgt, context)   # out: (batch, tgt_len, hidden)
        return self.output_layer(out)          # (batch, tgt_len, output_size)

    def forward(self, src, tgt):
        context = self.encode(src)
        return self.decode(tgt, context)

n_input, n_hidden, n_output = 1, 16, 1
model = EncoderDecoder(n_input, n_hidden, n_output)

enc_params = sum(p.numel() for p in model.encoder.parameters())
dec_params = sum(p.numel() for p in model.decoder.parameters())
out_params = sum(p.numel() for p in model.output_layer.parameters())

print(f'Encoder parameters: {enc_params}')
print(f'Decoder parameters: {dec_params}')
print(f'Output layer:       {out_params}')
print(f'Total:              {enc_params + dec_params + out_params}')
print()

# Verify shapes with a dummy forward pass
src = torch.randn(1, 3, n_input)   # batch=1, src_len=3, input=1
tgt = torch.randn(1, 2, n_output)  # batch=1, tgt_len=2, output=1
out = model(src, tgt)
print(f'src shape: {src.shape}  →  output shape: {out.shape}')
print('Input length 3, output length 2 — different lengths confirmed')

Encoder parameters: 304
Decoder parameters: 304
Output layer:       17
Total:              625

src shape: torch.Size([1, 3, 1])  →  output shape: torch.Size([1, 2, 1])
Input length 3, output length 2 — different lengths confirmed


## Part 3: Sequence Reversal Task

Training the encoder-decoder to reverse a 4-element sequence.
Input: $(x_1, x_2, x_3, x_4)$ → Output: $(x_4, x_3, x_2, x_1)$.

This is a minimal seq2seq task that requires the encoder to compress the full input into $c$ and the decoder to reconstruct it in reverse order — a direct test of the context vector.

In [5]:
# Training data: 50 random sequences of length 4
torch.manual_seed(42)
N = 200
src_data = torch.randn(N, 4, 1)          # (N, 4, 1)
tgt_data = src_data.flip(dims=[1])        # reversed

# Teacher forcing: decoder input is shifted target (start token = 0)
start = torch.zeros(N, 1, 1)
dec_input = torch.cat([start, tgt_data[:, :-1, :]], dim=1)  # (N, 4, 1)

print(f'Source shape:        {src_data.shape}')
print(f'Target shape:        {tgt_data.shape}')
print(f'Decoder input shape: {dec_input.shape}')
print()
print('Example (first sequence):')
print(f'  src:       {src_data[0].squeeze().tolist()}')
print(f'  target:    {tgt_data[0].squeeze().tolist()}')
print(f'  dec input: {dec_input[0].squeeze().tolist()}')

Source shape:        torch.Size([200, 4, 1])
Target shape:        torch.Size([200, 4, 1])
Decoder input shape: torch.Size([200, 4, 1])

Example (first sequence):
  src:       [1.9269152879714966, 1.4872840642929077, 0.9007171988487244, -2.1055209636688232]
  target:    [-2.1055209636688232, 0.9007171988487244, 1.4872840642929077, 1.9269152879714966]
  dec input: [0.0, -2.1055209636688232, 0.9007171988487244, 1.4872840642929077]


In [6]:
torch.manual_seed(0)
model = EncoderDecoder(input_size=1, hidden_size=32, output_size=1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

for epoch in range(1, 501):
    pred = model(src_data, dec_input)          # (N, 4, 1)
    loss = criterion(pred, tgt_data)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        print(f'Epoch {epoch:4d}  loss={loss.item():.6f}')

Epoch  100  loss=0.028884


Epoch  200  loss=0.013643


Epoch  300  loss=0.007670


Epoch  400  loss=0.004708
Epoch  500  loss=0.002989


In [7]:
# Evaluate on a new sequence (no teacher forcing — autoregressive)
model.eval()
with torch.no_grad():
    test_src = torch.tensor([[[0.5], [-0.3], [0.8], [-0.1]]])  # (1, 4, 1)
    expected = test_src.flip(dims=[1])

    # Autoregressive decoding
    context = model.encode(test_src)
    dec_in  = torch.zeros(1, 1, 1)   # start token
    outputs = []
    for _ in range(4):
        out = model.decode(dec_in, context)   # (1, 1, 1)
        outputs.append(out.item())
        dec_in = out

print('Autoregressive decoding on test sequence:')
print(f'  Input:    {test_src.squeeze().tolist()}')
print(f'  Expected: {expected.squeeze().tolist()}')
print(f'  Predicted:{[round(v, 4) for v in outputs]}')

Autoregressive decoding on test sequence:
  Input:    [0.5, -0.30000001192092896, 0.800000011920929, -0.10000000149011612]
  Expected: [-0.10000000149011612, 0.800000011920929, -0.30000001192092896, 0.5]
  Predicted:[-0.1059, -0.1277, -0.1322, -0.1331]


## Summary

What this notebook demonstrated:

1. **Manual forward pass** — reproduced the Note 26 worked example. The encoder processes $(1.0, 0.5, -1.0)$ using Note 24 weights, producing $c = [-0.3685, -0.0137]^T$. The decoder is initialised with $c$ and generates $\hat{y}_1 = -0.1016$, $\hat{y}_2 = -0.0459$ — output length 2 from input length 3.
2. **PyTorch architecture** — encoder and decoder are separate `nn.RNN` modules. The context vector is the encoder's final hidden state `h_n`, passed as the decoder's initial hidden state. Input and output lengths are independent.
3. **Sequence reversal** — trained the encoder-decoder to reverse a 4-element sequence using teacher forcing. After 500 epochs the model learns to reverse sequences accurately, confirming that the context vector carries sufficient information for this short-sequence task.